# Triage trajectories in the LLM arms

Summarize pre-to-post movement between appropriate, over-, and under-triage states.

## Configure analysis

In [ ]:
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.path import Path as MplPath

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
FIG_DIR = ROOT / "manuscript" / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

VIGNETTES = ["A", "B", "C", "D", "E"]
VIGNETTE_LABELS = {
    "A": "A — Psychosis",
    "B": "B — Panic attack",
    "C": "C — OCD",
    "D": "D — Bipolar / mania",
    "E": "E — Control case",
}
LLM_ARMS = [1, 2, 3]
STATE_ORDER = ["correct", "over", "under"]
STATE_LABELS = {
    "correct": "Appropriate",
    "over": "Overtriage",
    "under": "Undertriage",
}
STATE_COLORS = {
    "correct": "#009E73",
    "over": "#D55E00",
    "under": "#0072B2",
}

BAR_X_PRE = 0.08
BAR_X_POST = 0.92
BAR_W = 0.07
FLOW_X_PRE = BAR_X_PRE + BAR_W
FLOW_X_POST = BAR_X_POST
STATE_GAP = 0.025

## Load LLM-arm data

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

needed = ["prolific_id", "study_arm"]
for vignette in VIGNETTES:
    for phase in ("pre", "post"):
        needed.extend(
            [
                f"vig_{vignette}_{phase}_nextstep_correct",
                f"vig_{vignette}_{phase}_nextstep_over",
                f"vig_{vignette}_{phase}_nextstep_under",
            ]
        )

df = pd.read_excel(DATA_PATH)
missing = [col for col in needed if col not in df.columns]
if missing:
    raise KeyError(f"Columns not found: {missing}")

llm_df = df[df["study_arm"].isin(LLM_ARMS)].copy()
print(f"Analysis sample (LLM arms pooled): {len(llm_df):,} participants")
print(llm_df["study_arm"].value_counts().sort_index())

## Classify triage states and transitions

In [ ]:
def classify_triage_state(correct, over, under) -> str | float:
    if pd.isna(correct):
        return np.nan
    if correct == 1:
        return "correct"
    if pd.isna(over) or pd.isna(under):
        return np.nan
    if over == 1:
        return "over"
    if under == 1:
        return "under"
    return np.nan


def triage_columns(vignette: str, phase: str) -> tuple[str, str, str]:
    prefix = f"vig_{vignette}_{phase}_nextstep"
    return f"{prefix}_correct", f"{prefix}_over", f"{prefix}_under"


rows = []
for _, participant in llm_df.iterrows():
    for vignette in VIGNETTES:
        pre_cols = triage_columns(vignette, "pre")
        post_cols = triage_columns(vignette, "post")
        pre_state = classify_triage_state(
            participant[pre_cols[0]], participant[pre_cols[1]], participant[pre_cols[2]]
        )
        post_state = classify_triage_state(
            participant[post_cols[0]], participant[post_cols[1]], participant[post_cols[2]]
        )
        rows.append(
            {
                "prolific_id": participant["prolific_id"],
                "study_arm": participant["study_arm"],
                "vignette": vignette,
                "pre_state": pre_state,
                "post_state": post_state,
            }
        )

long_df = pd.DataFrame(rows)
long_df = long_df.dropna(subset=["pre_state", "post_state"])
long_df["transition"] = long_df["pre_state"] + " → " + long_df["post_state"]
print(f"Participant × vignette rows with complete pre/post state: {len(long_df):,}")
print(long_df.groupby("vignette").size().to_string())

## Create transition tables

In [ ]:
def transition_table(vignette: str) -> pd.DataFrame:
    sub = long_df[long_df["vignette"] == vignette]
    counts = pd.crosstab(sub["pre_state"], sub["post_state"])
    counts = counts.reindex(index=STATE_ORDER, columns=STATE_ORDER, fill_value=0)
    counts = counts.loc[counts.sum(axis=1) > 0, counts.sum(axis=0) > 0]

    row_pct = counts.div(counts.sum(axis=1), axis=0).mul(100).round(1)
    total_pct = counts / len(sub) * 100

    display_rows = []
    for pre in counts.index:
        for post in counts.columns:
            n = int(counts.loc[pre, post])
            if n == 0:
                continue
            display_rows.append(
                {
                    "vignette": vignette,
                    "pre_state": STATE_LABELS[pre],
                    "post_state": STATE_LABELS[post],
                    "n": n,
                    "pct_of_pre_row": row_pct.loc[pre, post],
                    "pct_of_vignette_n": round(total_pct.loc[pre, post], 1),
                }
            )
    return pd.DataFrame(display_rows)


transition_tables = {v: transition_table(v) for v in VIGNETTES}

for vignette in VIGNETTES:
    n = (long_df["vignette"] == vignette).sum()
    print(f"\n{'=' * 72}")
    print(f"{VIGNETTE_LABELS[vignette]}  (n = {n})")
    print("=" * 72)
    table = transition_tables[vignette]
    if table.empty:
        print("No transitions.")
        continue
    print(
        table.to_string(
            index=False,
            formatters={
                "pct_of_pre_row": lambda x: f"{x:.1f}%",
                "pct_of_vignette_n": lambda x: f"{x:.1f}%",
            },
        )
    )

    counts = pd.crosstab(
        long_df.loc[long_df["vignette"] == vignette, "pre_state"],
        long_df.loc[long_df["vignette"] == vignette, "post_state"],
    )
    counts = counts.reindex(index=STATE_ORDER, columns=STATE_ORDER, fill_value=0)
    counts = counts.loc[counts.sum(axis=1) > 0, counts.sum(axis=0) > 0]
    counts.index = [STATE_LABELS[s] for s in counts.index]
    counts.columns = [STATE_LABELS[s] for s in counts.columns]
    print("\nCounts (pre × post):")
    print(counts.to_string())

## Plot triage transitions by vignette

In [ ]:
def crosstab_for(vignette: str) -> pd.DataFrame:
    sub = long_df[long_df["vignette"] == vignette]
    tab = pd.crosstab(sub["pre_state"], sub["post_state"])
    return tab.reindex(index=STATE_ORDER, columns=STATE_ORDER, fill_value=0)


def stack_positions(totals: pd.Series) -> dict[str, tuple[float, float]]:
    active = [state for state in STATE_ORDER if totals.get(state, 0) > 0]
    if not active:
        return {}
    n = totals[active].sum()
    y = 0.04
    usable = 0.92
    gap_total = STATE_GAP * max(len(active) - 1, 0)
    positions = {}
    for state in active:
        height = (totals[state] / n) * (usable - gap_total)
        positions[state] = (y, y + height)
        y += height + STATE_GAP
    return positions


def flow_ribbons(tab: pd.DataFrame) -> tuple[list[dict], dict, dict]:
    pre_totals = tab.sum(axis=1)
    post_totals = tab.sum(axis=0)
    pre_stack = stack_positions(pre_totals)
    post_stack = stack_positions(post_totals)
    pre_cursor = {state: pre_stack[state][0] for state in pre_stack}
    post_cursor = {state: post_stack[state][0] for state in post_stack}
    ribbons = []
    for pre in STATE_ORDER:
        if pre not in pre_stack or pre_totals[pre] == 0:
            continue
        for post in STATE_ORDER:
            if post not in post_stack or post_totals[post] == 0:
                continue
            value = int(tab.loc[pre, post])
            if value == 0:
                continue
            sh = (pre_stack[pre][1] - pre_stack[pre][0]) * value / pre_totals[pre]
            th = (post_stack[post][1] - post_stack[post][0]) * value / post_totals[post]
            ribbons.append(
                {
                    "pre": pre,
                    "post": post,
                    "value": value,
                    "y0b": pre_cursor[pre],
                    "y0t": pre_cursor[pre] + sh,
                    "y1b": post_cursor[post],
                    "y1t": post_cursor[post] + th,
                }
            )
            pre_cursor[pre] += sh
            post_cursor[post] += th
    return ribbons, pre_stack, post_stack


def add_bezier_ribbon(ax, x0, x1, y0b, y0t, y1b, y1t, color, alpha=0.45):
    xm = (x0 + x1) / 2
    verts = [
        (x0, y0b),
        (xm, y0b),
        (xm, y1b),
        (x1, y1b),
        (x1, y1t),
        (xm, y1t),
        (xm, y0t),
        (x0, y0t),
        (x0, y0b),
    ]
    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]
    ax.add_patch(
        mpatches.PathPatch(
            MplPath(verts, codes),
            facecolor=color,
            edgecolor="none",
            alpha=alpha,
            linewidth=0,
            zorder=1,
        )
    )


def draw_flow_panel(ax, vignette: str) -> None:
    tab = crosstab_for(vignette)
    n = int(tab.values.sum())
    ribbons, pre_stack, post_stack = flow_ribbons(tab)

    for rib in ribbons:
        add_bezier_ribbon(
            ax,
            FLOW_X_PRE,
            FLOW_X_POST,
            rib["y0b"],
            rib["y0t"],
            rib["y1b"],
            rib["y1t"],
            STATE_COLORS[rib["pre"]],
        )

    for state, (y0, y1) in pre_stack.items():
        ax.add_patch(
            mpatches.FancyBboxPatch(
                (BAR_X_PRE, y0),
                BAR_W,
                y1 - y0,
                boxstyle="square,pad=0",
                facecolor=STATE_COLORS[state],
                edgecolor="white",
                linewidth=1.2,
                zorder=3,
            )
        )
        ax.text(
            BAR_X_PRE - 0.015,
            (y0 + y1) / 2,
            f"{STATE_LABELS[state]}\n(n={int(tab.sum(axis=1)[state])})",
            ha="right",
            va="center",
            fontsize=8,
            color=STATE_COLORS[state],
            fontweight="bold",
            linespacing=1.15,
        )

    for state, (y0, y1) in post_stack.items():
        ax.add_patch(
            mpatches.FancyBboxPatch(
                (BAR_X_POST, y0),
                BAR_W,
                y1 - y0,
                boxstyle="square,pad=0",
                facecolor=STATE_COLORS[state],
                edgecolor="white",
                linewidth=1.2,
                zorder=3,
            )
        )
        ax.text(
            BAR_X_POST + BAR_W + 0.015,
            (y0 + y1) / 2,
            f"{STATE_LABELS[state]}\n(n={int(tab.sum(axis=0)[state])})",
            ha="left",
            va="center",
            fontsize=8,
            color=STATE_COLORS[state],
            fontweight="bold",
            linespacing=1.15,
        )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(f"{VIGNETTE_LABELS[vignette]}\n(n = {n})", fontsize=10, fontweight="bold", pad=8)
    ax.text(0.22, 1.02, "Pre", transform=ax.transAxes, ha="center", va="bottom", fontsize=9, color="#444444")
    ax.text(0.78, 1.02, "Post", transform=ax.transAxes, ha="center", va="bottom", fontsize=9, color="#444444")


plt.rcParams.update({"font.family": "sans-serif", "font.size": 9})
fig, axes = plt.subplots(2, 3, figsize=(14, 8.5), constrained_layout=True)
axes_flat = axes.flatten()

for idx, vignette in enumerate(VIGNETTES):
    draw_flow_panel(axes_flat[idx], vignette)
axes_flat[5].axis("off")

legend_handles = [
    mpatches.Patch(facecolor=STATE_COLORS[state], edgecolor="white", label=STATE_LABELS[state])
    for state in STATE_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
    fontsize=9,
)
fig.suptitle(
    "",
    fontsize=12,
    fontweight="bold",
    y=1.02,
)

plt.show()

FIG_PNG = FIG_DIR / "Figure_triage_llm_trajectories.png"
FIG_PDF = FIG_DIR / "Figure_triage_llm_trajectories.pdf"

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
print(f"Saved {FIG_PNG}")
print(f"Saved {FIG_PDF}")